In [1]:
import torch
from torch.utils.data import DataLoader, random_split
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
import os
import random

from data.preprocessing.pipeline import Pipeline
from data.datasets.MLP_Dataset import MoleculeDataset
from models.MLP import CSMLP
from models.MLP import FocalLoss
from utils.paths import WEIGHTS_DIR

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

In [3]:
def compute_class_weights(train_df):

    class_counts = np.bincount(train_df["taste_cluster"].values)
    class_weights = 1.0 / class_counts
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    class_weights = torch.tensor(class_weights, dtype=torch.float32)

    print("Class counts:", class_counts)
    print("Class weights:", class_weights)

    return torch.tensor(class_weights, dtype=torch.float32)

In [4]:
random_state = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 20
batch_size=64,
lr=1e-3,
dropout=0.3,
val_size=0.2,
patience=5,
checkpoint_path="best_model.pth"
save_dir_path = os.path.join(WEIGHTS_DIR, 'MLP')

set_seed(random_state)

In [5]:
pipeline = Pipeline(feature_type="morgan", download=False)

Extracting raw features:   0%|          | 0/2064 [00:00<?, ?it/s]

Extracting raw features: 100%|██████████| 888/888 [00:04<00:00, 199.67it/s]


In [6]:
pipeline.get_processed_data()[0]

,Pubchem_ID,taste_cluster,SMILES,feat_0,feat_1,feat_2,feat_3,feat_4,feat_5,feat_6,...,feat_2038,feat_2039,feat_2040,feat_2041,feat_2042,feat_2043,feat_2044,feat_2045,feat_2046,feat_2047
0,95609,0,CCC(C(=O)CC)O,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,104224,0,CCCCCCCCCCC(OC)OC,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,5284499,6,CCCCCCCC/C=C\CCCCCCCCO,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,10886,4,CCCCCC(=O)OCCCCC,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,12978217,3,C/C=C/C(=O)[C@@H]1[C@H](C=CCC1(C)C)C,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2059,229385,1,CCCCCCCCCCOC(=O)CC,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2060,62387,2,CC1(C2CCC1([C@H](C2)OC=O)C)C,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
2061,8878,7,CCCCCCC(=O)OCC=C,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2062,61262,1,CC(=O)OCC1=CCC2CC1C2(C)C,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [7]:
def train_model(
    train_df,
    epochs=20,
    batch_size=64,
    lr=1e-3,
    dropout=0.3,
    val_size=0.2,
    patience=5,
    hidden_dims=[800, 300, 150],
    checkpoint_path="best_model.pth",
    save_dir_path="."
):
    set_seed()
    os.makedirs(save_dir_path, exist_ok=True)

    full_dataset = MoleculeDataset(train_df, train=True)
    n_total = len(full_dataset)
    n_val = int(n_total * val_size)
    n_train = n_total - n_val

    train_subset, val_subset = random_split(full_dataset, [n_train, n_val])

    train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False)

    model = CSMLP(train_subset[0][0].shape[0], hidden_dims=hidden_dims, dropout=dropout, output_dim=8).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLoss(gamma=1.5, weight=compute_class_weights(train_df=train_df).to(device))

    best_val_loss = float("inf")
    patience_counter = 0

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(epochs):
        model.train()
        train_losses = []

        for x, y in train_loader:
            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # -------------------------
        # Validation
        # -------------------------
        model.eval()
        val_losses = []
        all_preds = []
        all_targets = []

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device)
                y = y.to(device)

                logits = model(x)
                loss = criterion(logits, y)
                val_losses.append(loss.item())

                preds = torch.argmax(logits, dim=1)

                all_preds.append(preds.cpu().numpy())
                all_targets.append(y.cpu().numpy())

        train_loss = np.mean(train_losses)
        val_loss = np.mean(val_losses)

        print(f"Epoch {epoch+1}/{epochs} | train_loss={train_loss:.5f} | val_loss={val_loss:.5f}")

        # -------------------------
        # EarlyStopping + checkpoint
        # -------------------------
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0

            torch.save(model.state_dict(), os.path.join(save_dir_path, checkpoint_path))
            print(f"  → New best model saved! val_loss={best_val_loss:.5f}")

        else:
            patience_counter += 1
            print(f"  (patience {patience_counter}/{patience})")

            if patience_counter >= patience:
                print("\nEarly stopping triggered!")
                break

    # -------------------------
    # final validation F1 (macro)
    # -------------------------
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    print("\nFinal macro F1 on validation:", macro_f1)

    return model, macro_f1, best_val_loss

In [8]:
def predict(model, test_df, batch_size=64):
    test_dataset = MoleculeDataset(test_df, train=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model.eval()
    all_ids = []
    all_preds = []

    with torch.no_grad():
        for x, ids in test_loader:
            x = x.to(device)
            logits = model(x)
            
            preds = torch.argmax(logits, dim=1)

            all_ids.extend(ids)
            all_preds.extend(preds.cpu().numpy())

    pred_df = pd.DataFrame({"Pubchem_ID": all_ids, "taste_cluster": all_preds})
    return pred_df

In [9]:
train_df, test_df = pipeline.get_processed_data()

model, f1, best_loss = train_model(
    train_df,
    epochs=100,
    batch_size=64,
    lr=1e-4,
    dropout=0.3,
    patience=25,
    hidden_dims=[800],
    checkpoint_path=checkpoint_path,
    save_dir_path=save_dir_path
)

Class counts: [530 296 220 402 286  43  85 202]
Class weights: tensor([0.2707, 0.4847, 0.6521, 0.3569, 0.5016, 3.3362, 1.6877, 0.7102])
Epoch 1/100 | train_loss=0.70011 | val_loss=0.71948
  → New best model saved! val_loss=0.71948


/tmp/ipykernel_35205/1475540974.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(class_weights, dtype=torch.float32)


Epoch 2/100 | train_loss=0.64862 | val_loss=0.68069
  → New best model saved! val_loss=0.68069
Epoch 3/100 | train_loss=0.60172 | val_loss=0.64558
  → New best model saved! val_loss=0.64558
Epoch 4/100 | train_loss=0.56077 | val_loss=0.61551
  → New best model saved! val_loss=0.61551
Epoch 5/100 | train_loss=0.52775 | val_loss=0.58886
  → New best model saved! val_loss=0.58886
Epoch 6/100 | train_loss=0.49526 | val_loss=0.56437
  → New best model saved! val_loss=0.56437
Epoch 7/100 | train_loss=0.46235 | val_loss=0.54796
  → New best model saved! val_loss=0.54796
Epoch 8/100 | train_loss=0.43465 | val_loss=0.53390
  → New best model saved! val_loss=0.53390
Epoch 9/100 | train_loss=0.40796 | val_loss=0.52029
  → New best model saved! val_loss=0.52029
Epoch 10/100 | train_loss=0.38586 | val_loss=0.51008
  → New best model saved! val_loss=0.51008
Epoch 11/100 | train_loss=0.36468 | val_loss=0.50332
  → New best model saved! val_loss=0.50332
Epoch 12/100 | train_loss=0.34802 | val_loss=0.4

In [10]:
model.load_state_dict(torch.load(os.path.join(save_dir_path, checkpoint_path)))
pred = predict(model, test_df)

In [11]:
pred

,Pubchem_ID,taste_cluster
0,tensor(32594),2
1,tensor(7130),2
2,tensor(7455),1
3,tensor(31226),7
4,tensor(17000),0
...,...,...
883,tensor(26334),0
884,tensor(104421),2
885,tensor(21263168),6
886,tensor(5368236),4
